<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week6_inclass_tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 6 – In-class tasks

Work through these during the workshop, alongside the slides. The setup cell loads three datasets:

- `student_summary` – our class, one row per student: mean reaction time (`mean_rt`), finger length (`finger_mm`, with Week 3's corrections), device and caffeine
- `hotels` – the 207 Vienna hotels from Weeks 4 and 5
- `cps_df` – the textbook's CPS earnings data, 18,241 people with a graduate degree

Use `cov_type='HC1'` for every model.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

base = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/"

# Our class: one row per student
df = pd.read_csv(base + "week3_reaction_times.csv")
df['finger_mm'] = df['finger_length_corrected']
student_summary = (
    df.groupby('name')
    .agg(mean_rt=('reaction_time_ms', 'mean'), device=('device', 'first'),
         caffeine=('caffeine', 'first'), finger_mm=('finger_mm', 'first'))
    .reset_index()
)

# Vienna hotels, same filters as Weeks 4 and 5
hotels = pd.read_csv(base + "hotels_vienna.csv")
hotels = hotels[(hotels['accommodation_type'] == 'Hotel') & (hotels['city_actual'] == 'Vienna')
                & hotels['stars'].between(3, 4) & (hotels['price'] <= 600)].copy()
hotels['ln_price'] = np.log(hotels['price'])

# CPS earnings: graduate-degree holders
cps_df = pd.read_csv(base + "cps_gender_pay_gap.csv")

print(len(student_summary), "students,", len(hotels), "hotels,", len(cps_df), "people")

## Task 1 – watch OVB happen

A made-up case where we know the truth: the true effect of our own price on sales is −3. Run the simulation, then fit three regressions: `sales` on `our_price`; `sales` on `our_price` and `competitor_price`; and `competitor_price` on `our_price`. Check that β̂ − β̂₁ = δ̂ × β̂₂.

Then change the `0.8` to `0`, run it all again, and see what happens to the bias.

In [ ]:
rng = np.random.default_rng(42)
competitor_price = rng.normal(0, 2.0, 300)
our_price = 0.8 * competitor_price + rng.normal(0, 1.2, 300)
sales = -3*our_price + 3*competitor_price + rng.normal(0, 2.5, 300)
sim = pd.DataFrame({'sales': sales, 'our_price': our_price, 'competitor_price': competitor_price})


## Task 2 – does our own coefficient move?

**Business question:** does the finger-length coefficient change once we control for device and caffeine?

Fit two models on `student_summary`: one with finger length alone, one adding device and caffeine. Compare the finger-length coefficient across both.

## Task 3 – testing a group of coefficients

In Week 5's online session the hotels spline had knots at 1 and 4 miles. Make the two spline variables, fit `price ~ distance + beyond_1 + beyond_4`, and test whether both spline terms are zero together with `model.f_test('beyond_1 = 0, beyond_4 = 0')`. Then test `beyond_4` on its own, and compare the R² with and without it.

```python
hotels['beyond_1'] = np.maximum(hotels['distance'] - 1, 0)
```

## Task 4 – the gender pay gap

**Business question:** how big is the gender pay gap among people with a graduate degree, and does it change once we compare people with the same degree?

Regress `ln_wage` on `female`. Then add education as a categorical variable, once with MA as the reference category and once with PhD: `C(education, Treatment("MA"))`. What happens to the female coefficient, and to the education coefficients?

## Task 5 – does the pattern differ by device?

**Business question:** does the association between finger length and reaction time depend on which device someone used?

Answer this with `smf.ols` on `student_summary`. Work out the formula string yourselves.